# Demand Forecasting — Phase 5

Trains a **Linear Regression** and a **Random Forest Regressor** to predict
per-cell delivery demand from time and zone features (mirroring the
Bike Sharing Demand dataset structure). Reports **MAE** and **RMSE**,
saves figures to `report/figures/`, and applies the best model back
to the simulation grid.

In [ ]:
import sys
from pathlib import Path

# Add project root to path so src.* imports resolve from any CWD
root = Path.cwd()
for d in [root, *root.parents]:
    if (d / 'src' / 'ml_pipeline.py').is_file():
        root = d
        break
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from src.ml_pipeline import (
    generate_demand_dataset,
    train_demand_models,
    print_demand_metrics,
    plot_demand_results,
    demand_forecast_for_grid,
    DEMAND_FEATURES,
)
print('Imports OK')

## 1. Dataset

Synthetic dataset mirroring **Bike Sharing Demand** features:
hour, day_of_week, temperature, weather, zone_type, density, is_hub.
Demand is computed with a realistic formula (peaks at midday, weekdays,
mild weather) plus Gaussian noise.

In [ ]:
df = generate_demand_dataset(n_samples=800, seed=42)
print(f'Shape: {df.shape}')
display(df.head())
display(df.describe().round(2))

In [ ]:
# Demand distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['demand'], bins=20, color='#1d3557', edgecolor='white')
axes[0].set_xlabel('Demand')
axes[0].set_ylabel('Count')
axes[0].set_title('Demand distribution')

avg_by_hour = df.groupby('hour')['demand'].mean()
axes[1].plot(avg_by_hour.index, avg_by_hour.values, marker='o', color='#e63946')
axes[1].set_xlabel('Hour of day')
axes[1].set_ylabel('Avg demand')
axes[1].set_title('Average demand by hour (shows midday peak)')
plt.tight_layout()
plt.show()

## 2. Model Training — 80/20 split

In [ ]:
results = train_demand_models(df)
print_demand_metrics(results)

## 3. Figures

Saved to `report/figures/demand_actual_vs_pred.png` and
`report/figures/demand_feature_importance.png`.

In [ ]:
plot_demand_results(results, show=True)

## 4. Apply forecast to simulation grid

The Random Forest model predicts a demand value for every cell based on
its zone type, density, hub status, and the current time context.
This links the ML output back into the simulation.

In [ ]:
import random
from src.grid_model import Grid, Zone
from src.visualization import plot_demand_heatmap

random.seed(42)
grid = Grid(10, 10, zone_limits={
    Zone.RESIDENTIAL: 10, Zone.COMMERCIAL: 10,
    Zone.HOSPITAL: 3,     Zone.SCHOOL: 5,
    Zone.INDUSTRIAL: 8,
})
grid.populate_grid()

print('Before forecast:')
demands_before = [cell.demand for row in grid.grid for cell in row]
print(f'  mean={sum(demands_before)/len(demands_before):.1f}  min={min(demands_before)}  max={max(demands_before)}')

demand_forecast_for_grid(
    grid,
    results['Random Forest']['model'],
    hour=12, day_of_week=1, temperature=25.0, weather=0,
)

demands_after = [cell.demand for row in grid.grid for cell in row]
print('After forecast (noon, weekday, 25C, clear):')
print(f'  mean={sum(demands_after)/len(demands_after):.1f}  min={min(demands_after)}  max={max(demands_after)}')

plot_demand_heatmap(grid, title='ML-forecast demand (noon, weekday)', show=True)

## Summary

| Model | MAE | RMSE |
|---|---|---|
| Linear Regression | see above | see above |
| Random Forest | see above | see above |

Random Forest outperforms Linear Regression because demand has non-linear
interactions (e.g. hub + commercial + midday all amplify demand together).
Feature importance shows `density` and `hour` as the top drivers.